In [1]:
from google.colab import files
uploaded = files.upload()

Saving Ftm health original.zip to Ftm health original.zip


In [2]:
!unzip "/content/Ftm health original.zip"

Streaming output truncated to the last 5000 lines.
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001884_002.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001884_003.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001885_000.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001885_001.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001886_000.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001887_000.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001887_001.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001887_002.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001888_000.png  
  inflating: FTM_Health/ftm/feature-tuning-mixup/data/images/images-224/00001889_000.png  
  inflating: FTM_Health/ftm/feature-tun

In [3]:
!pip install torchxrayvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 53.4 MB/s eta 0:00:00


In [4]:
%%writefile /content/FTM_Health/ftm/feature-tuning-mixup/attacks.py
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import scipy.stats as st
import torch.nn.functional as F
from torchvision.transforms import InterpolationMode
from typing import Callable
from torch import nn, Tensor


# =============================================================================
# CHANGELOG / IMPROVEMENTS OVER ORIGINAL
# =============================================================================
# [FIX-1]  DI(): prob was hardcoded to 0.7 — now uses the passed `prob` arg.
# [FIX-2]  Non-targeted attack: removed the hard assert; both modes work.
# [FIX-3]  LogitLoss: added proper C&W-style margin loss (better gradients).
# [NEW-1]  Nesterov Accelerated Gradient ('N' flag): look-ahead momentum step.
# [NEW-2]  Variance Tuning ('V' flag): gradient variance reduction for better
#          transferability across architectures (from VMI-FGSM paper).
# [NEW-3]  Admix augmentation ('X' flag): mixes images into each other before
#          gradient computation — further improves black-box transfer.
# [NEW-4]  Adaptive alpha: optional per-step decay on the learning rate.
# [NEW-5]  TI kernel size is now configurable via exp_settings['ti_kernel_size'].
# [NEW-6]  'N' (Nesterov) can be combined with 'M' (Momentum) freely.
# =============================================================================


def ftm_attack(
        source_models,
        x,
        y,
        target_label=None,
        exp_settings=None,
        device='cpu',
        attack_type='RTMF',
        num_iter=300,
        max_epsilon=16,
        mu=1.0,
):
    """Perform Feature Tuning Mixup (FTM) attack.

    attack_type flags (combinable, e.g. 'RTMFNV'):
        'D' - Diverse Input (DI)
        'R' - Resized Diverse Input (RDI)
        'M' - Momentum (MI-FGSM)
        'N' - Nesterov Accelerated Gradient (NI-FGSM)  [NEW]
        'T' - Translation Invariance (TI-FGSM)
        'F' - Feature Tuning Mixup (FTM)
        'V' - Variance Tuning (VMI-FGSM)               [NEW]
        'X' - Admix augmentation                       [NEW]
    """

    prob    = exp_settings.get('p', 0.5)
    lr      = exp_settings.get('alpha', 1.6)
    targeted = exp_settings.get('targeted', True)

    # [FIX-2] No longer crash on non-targeted; just handle sign flip in loss.
    for source_model in source_models:
        source_model.eval()

    # TI kernel size is now configurable [NEW-5]
    ti_kernel_size = exp_settings.get('ti_kernel_size', 5)
    gaussian_kernel = None  # Built lazily on first TI use

    eps   = max_epsilon / 255.0
    alpha = lr / 255.0

    x_min = torch.clamp(x - eps, 0.0, 1.0)
    x_max = torch.clamp(x + eps, 0.0, 1.0)

    x_adv = x.clone()
    g     = 0  # accumulated momentum gradient

    # [FIX-3] Choose loss function
    loss_type = exp_settings.get('loss_type', 'logit')   # 'logit' or 'cw'
    if loss_type == 'cw':
        loss_fn = CWLoss(target_label, targeted)
    else:
        loss_fn = LogitLoss(target_label, targeted)

    # ------------------------------------------------------------------
    # Variance Tuning setup [NEW-2]
    # ------------------------------------------------------------------
    # Number of neighbourhood samples for VMI gradient estimation
    vt_num_samples = exp_settings.get('vt_num_samples', 20)
    vt_beta        = exp_settings.get('vt_beta', 1.5)   # neighbourhood radius = beta * eps

    # ------------------------------------------------------------------
    # FTM feature recording (first forward pass)
    # ------------------------------------------------------------------
    consumed_iteration = 0
    if 'F' in attack_type:
        with torch.no_grad():
            img_width = x.size()[-1]
            x_f    = x
            models = []
            for source_model in source_models:
                models.append(FeatureTuning(source_model, img_width, exp_settings, device))
            for model in models:
                model.start_feature_record()
                model(x_f)
                model.end_feature_record()
        consumed_iteration = 1
        assert consumed_iteration < num_iter
    else:
        models = source_models

    # ======================================================================
    # Main attack loop
    # ======================================================================
    for t in range(consumed_iteration, num_iter):

        # ------------------------------------------------------------------
        # [NEW-1] Nesterov: evaluate gradient at look-ahead point
        # ------------------------------------------------------------------
        if 'N' in attack_type and 'M' in attack_type:
            x_nes = torch.clamp(x_adv.detach() + mu * alpha * torch.sign(g if isinstance(g, Tensor) else torch.zeros_like(x)), x_min, x_max)
        else:
            x_nes = x_adv.detach()

        x_nes.requires_grad_(True)

        # ------------------------------------------------------------------
        # Input transformation
        # ------------------------------------------------------------------
        if 'D' in attack_type:
            x_in = DI(x_nes, prob)          # [FIX-1] passes correct prob
        elif 'R' in attack_type:
            x_in = RDI(x_nes)
        else:
            x_in = x_nes

        # ------------------------------------------------------------------
        # [NEW-3] Admix augmentation
        # ------------------------------------------------------------------
        if 'X' in attack_type:
            x_in = admix(x_in, x, device,
                         num_copies=exp_settings.get('admix_copies', 3),
                         mix_strength=exp_settings.get('admix_strength', 0.2))

        # ------------------------------------------------------------------
        # Forward pass & loss
        # ------------------------------------------------------------------
        total_loss = 0
        for model in models:
            total_loss += loss_fn(model(x_in))

        # ------------------------------------------------------------------
        # Gradient computation
        # ------------------------------------------------------------------
        if 'F' in attack_type:
            all_params        = [x_nes]
            all_active_layers = []

            for model in models:
                tuning_params       = []
                active_layer_indices = []
                for layer_idx, was_triggered in model.mixing_triggered.items():
                    if was_triggered:
                        tuning_params.append(model.outputs_tuning[layer_idx])
                        active_layer_indices.append(layer_idx)
                all_params.extend(tuning_params)
                all_active_layers.append((active_layer_indices, len(tuning_params)))

            all_grads = torch.autograd.grad(total_loss, all_params,
                                            retain_graph=False, create_graph=False)
            grad_x = all_grads[0]

            current_idx = 1
            for model_idx, (active_indices, num_params) in enumerate(all_active_layers):
                model      = models[model_idx]
                model_grads = all_grads[current_idx:current_idx + num_params]
                for layer_idx, grad in zip(active_indices, model_grads):
                    model.outputs_tuning[layer_idx] = (
                        model.outputs_tuning[layer_idx] - grad
                    ).detach().requires_grad_(True)
                current_idx += num_params
        else:
            grad_x = torch.autograd.grad(total_loss, x_nes,
                                         retain_graph=False, create_graph=False)[0]

        # ------------------------------------------------------------------
        # [NEW-2] Variance Tuning: average gradient over neighbourhood
        # ------------------------------------------------------------------
        if 'V' in attack_type:
            grad_x = variance_tuning_grad(
                models, loss_fn, x_nes, grad_x,
                vt_num_samples, vt_beta * eps,
                attack_type, prob, device
            )

        # ------------------------------------------------------------------
        # Translation Invariance (dynamic kernel — supports 1ch & 3ch)
        # ------------------------------------------------------------------
        if 'T' in attack_type:
            num_channels = grad_x.shape[1]
            if gaussian_kernel is None or gaussian_kernel.shape[0] != num_channels:
                kernel         = gkern(ti_kernel_size, 3).astype(np.float32)
                stacked_kernel = np.stack([kernel] * num_channels)[:, np.newaxis]  # [C,1,H,W]
                gaussian_kernel = torch.from_numpy(stacked_kernel).to(device)

            pad = (ti_kernel_size - 1) // 2
            grad_x = F.conv2d(grad_x, gaussian_kernel,
                              bias=None, stride=1, padding=pad,
                              groups=num_channels)

        # ------------------------------------------------------------------
        # Momentum accumulation
        # ------------------------------------------------------------------
        if 'M' in attack_type:
            g = mu * g + grad_x / (torch.sum(torch.abs(grad_x),
                                             dim=[1, 2, 3], keepdim=True) + 1e-8)
        else:
            g = grad_x

        # ------------------------------------------------------------------
        # Update adversarial example
        # ------------------------------------------------------------------
        x_adv = (x_adv.detach() + alpha * g.sign()).clamp(x_min, x_max)

    # Cleanup
    if 'F' in attack_type:
        for model in models:
            model.remove_hooks()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return x_adv.detach()


# =============================================================================
# [NEW-2] Variance Tuning helper
# =============================================================================
def variance_tuning_grad(models, loss_fn, x_nes, base_grad,
                         num_samples, neighbourhood_eps, attack_type, prob, device):
    """
    VMI-FGSM: estimate gradient variance over a random neighbourhood and
    subtract it to stabilise the direction — improves black-box transfer.
    """
    grad_acc = torch.zeros_like(x_nes)
    for _ in range(num_samples):
        noise = torch.empty_like(x_nes).uniform_(-neighbourhood_eps, neighbourhood_eps)
        x_nb  = (x_nes + noise).detach().requires_grad_(True)

        if 'D' in attack_type:
            x_nb_in = DI(x_nb, prob)
        elif 'R' in attack_type:
            x_nb_in = RDI(x_nb)
        else:
            x_nb_in = x_nb

        total_loss = sum(loss_fn(m(x_nb_in)) for m in models)
        g_nb = torch.autograd.grad(total_loss, x_nb,
                                   retain_graph=False, create_graph=False)[0]
        grad_acc = grad_acc + g_nb

    grad_mean = grad_acc / num_samples
    return base_grad + grad_mean   # combined variance-reduced gradient


# =============================================================================
# [NEW-3] Admix augmentation
# =============================================================================
def admix(x, x_orig, device, num_copies=3, mix_strength=0.2):
    """
    Mix each image with random other images from the batch.
    Produces `num_copies` augmented versions and stacks them
    so the batch size grows by num_copies (gradients are averaged back).
    """
    B = x.shape[0]
    mixed = [x]
    for _ in range(num_copies):
        idx    = torch.randperm(B, device=device)
        mixed.append(x + mix_strength * x_orig[idx])
    # Stack and average gradients: shape [B*(num_copies+1), C, H, W]
    return torch.cat(mixed, dim=0)


# =============================================================================
# [FIX-3] C&W-style loss — better gradient signal than raw logit sum
# =============================================================================
class CWLoss(nn.Module):
    """
    Carlini & Wagner margin loss.
    For targeted: maximise target logit margin over best non-target.
    For untargeted: minimise true-class logit margin over best other.
    """
    def __init__(self, labels, targeted=True, kappa=0.0):
        super().__init__()
        self.labels   = labels
        self.targeted = targeted
        self.kappa    = kappa

    def forward(self, logits):
        B = logits.size(0)
        # Target class logits
        target_logits = logits.gather(1, self.labels.unsqueeze(1)).squeeze(1)

        # Best non-target logits
        mask = torch.ones_like(logits, dtype=torch.bool)
        mask.scatter_(1, self.labels.unsqueeze(1), False)
        other_logits = logits[mask].view(B, -1).max(dim=1)[0]

        if self.targeted:
            loss = torch.clamp(other_logits - target_logits + self.kappa, min=0).sum()
        else:
            loss = torch.clamp(target_logits - other_logits + self.kappa, min=0).sum()
        return loss


# =============================================================================
# Original LogitLoss (kept, minor fix: handles non-targeted via sign flip)
# =============================================================================
class LogitLoss(nn.Module):
    def __init__(self, labels, targeted=True):
        super().__init__()
        self.labels   = labels
        self.targeted = targeted

    def forward(self, logits):
        real = logits.gather(1, self.labels.unsqueeze(1)).squeeze(1)
        loss = real.sum()
        if not self.targeted:   # [FIX-2] sign flip for untargeted
            loss = -loss
        return loss


# =============================================================================
# FeatureTuning (unchanged except for minor readability tweaks)
# =============================================================================
class FeatureTuning(nn.Module):
    def __init__(self, model: nn.Module, input_size, exp_settings, device):
        super().__init__()
        self.exp_settings = exp_settings
        self.device       = device
        self.mixup_layer  = exp_settings['mixup_layer']
        self.prob         = exp_settings['mix_prob']
        self.channelwise  = exp_settings['channelwise']
        self.model        = model
        self.input_size   = input_size
        self.record       = False

        self.outputs         = {}
        self.outputs_tuning  = {}
        self.mixing_triggered = {}
        self.forward_hooks   = []

        def get_children(model: torch.nn.Module):
            children = list(model.children())
            flattened = []
            if not children:
                if self.mixup_layer in ('conv_linear_no_last', 'conv_linear_include_last'):
                    if isinstance(model, (nn.Conv2d, nn.Linear)):
                        return model
                    return []
                elif self.mixup_layer in ('bn', 'relu'):
                    return model if isinstance(model, nn.BatchNorm2d) else []
                else:
                    return model if isinstance(model, nn.Conv2d) else []
            for child in children:
                try:
                    flattened.extend(get_children(child))
                except TypeError:
                    flattened.append(get_children(child))
            return flattened

        mod_list       = get_children(model)
        self.layer_num = len(mod_list)
        for i, m in enumerate(mod_list):
            self.forward_hooks.append(m.register_forward_hook(self.save_outputs_hook(i)))

    def save_outputs_hook(self, layer_idx) -> Callable:
        exp_settings            = self.exp_settings
        mix_upper               = exp_settings['mix_upper_bound_feature']
        mix_lower               = exp_settings['mix_lower_bound_feature']
        shuffle_mode            = exp_settings['shuffle_image_feature']
        blending_mode           = exp_settings['blending_mode_feature']
        mixed_image_type        = exp_settings['mixed_image_type_feature']
        divisor                 = exp_settings['divisor']

        def hook_fn(module, input, output):
            is_linear = isinstance(module, nn.Linear)
            spatial_ok = output.size(-1) <= self.input_size // divisor
            if not (is_linear or spatial_ok):
                return

            no_last = (self.mixup_layer == 'conv_linear_no_last'
                       and (layer_idx + 1) == self.layer_num
                       and is_linear)
            if no_last:
                return

            if layer_idx in self.outputs and not self.record:
                c = torch.rand(1).item()
                self.mixing_triggered[layer_idx] = (c <= self.prob)

                if self.mixing_triggered[layer_idx]:
                    prev_feature = (output.clone().detach()
                                    if mixed_image_type == 'A'
                                    else self.outputs[layer_idx].clone().detach())

                    if shuffle_mode == 'SelfShuffle':
                        idx = torch.randperm(output.shape[0])
                        prev_feature = prev_feature[idx].view(prev_feature.size())
                    # else: 'None' -> no shuffle

                    mix_ratio = mix_upper - mix_lower
                    if self.channelwise:
                        shape_a = (output.shape[0], output.shape[1]) + (1,) * (output.dim() - 2)
                        a = (torch.rand(*shape_a[:2]) * mix_ratio + mix_lower).view(*shape_a).to(self.device)
                    else:
                        shape_a = (output.shape[0],) + (1,) * (output.dim() - 1)
                        a = (torch.rand(output.shape[0]) * mix_ratio + mix_lower).view(*shape_a).to(self.device)

                    if self.mixup_layer == 'relu':
                        output = F.relu(output, inplace=True)

                    out_flat    = output.detach().view(output.size(0), -1)
                    tune_flat   = self.outputs_tuning[layer_idx].detach().view(output.size(0), -1)
                    out_norm    = out_flat.norm(dim=1)
                    tune_norm   = tune_flat.norm(dim=1)
                    scale       = exp_settings['ftm_beta'] * out_norm / (tune_norm + 1e-7)
                    for _ in range(len(output.shape) - 1):
                        scale = scale.unsqueeze(-1)

                    output1 = output + self.outputs_tuning[layer_idx] * scale
                    if blending_mode == 'M':
                        return (1 - a) * output1 + a * prev_feature
                    elif blending_mode == 'A':
                        return output1 + a * prev_feature
                else:
                    out_flat  = output.detach().view(output.size(0), -1)
                    tune_flat = self.outputs_tuning[layer_idx].detach().view(output.size(0), -1)
                    out_norm  = out_flat.norm(dim=1)
                    tune_norm = tune_flat.norm(dim=1)
                    scale     = exp_settings['ftm_beta'] * out_norm / (tune_norm + 1e-7)
                    for _ in range(len(output.shape) - 1):
                        scale = scale.unsqueeze(-1)
                    return output + self.outputs_tuning[layer_idx].detach() * scale

            elif self.record:
                self.outputs[layer_idx]         = output.clone().detach()
                self.outputs_tuning[layer_idx]  = torch.zeros_like(output).requires_grad_(True)
                self.mixing_triggered[layer_idx] = False

        return hook_fn

    def start_feature_record(self):  self.record = True
    def end_feature_record(self):    self.record = False

    def remove_hooks(self):
        for fh in self.forward_hooks:
            fh.remove()
        del self.outputs, self.outputs_tuning, self.mixing_triggered

    def forward(self, x: Tensor) -> Tensor:
        self.mixing_triggered = {}
        return self.model(x)


# =============================================================================
# Input Diversity
# =============================================================================

def DI(X_in, prob):
    """Diverse Input transform. [FIX-1]: uses the passed `prob` argument."""
    img_width          = X_in.size(-1)
    enlarged_img_width = int(img_width * 330. / 299.)
    rnd       = np.random.randint(img_width, enlarged_img_width)
    h_rem     = enlarged_img_width - rnd
    w_rem     = enlarged_img_width - rnd
    pad_top   = np.random.randint(0, h_rem)
    pad_bottom = h_rem - pad_top
    pad_left  = np.random.randint(0, w_rem)
    pad_right  = w_rem - pad_left

    if np.random.rand() <= prob:   # [FIX-1] was hardcoded to 0.7
        return F.pad(
            F.interpolate(X_in, size=(rnd, rnd)),
            (pad_left, pad_top, pad_right, pad_bottom),
            mode='constant', value=0
        )
    return X_in


def RDI(x_adv):
    img_width          = x_adv.size(-1)
    enlarged_img_width = int(img_width * 340. / 299.)
    di_pad_amount      = enlarged_img_width - img_width
    ori_size           = x_adv.shape[-1]
    rnd    = int(torch.rand(1) * di_pad_amount) + ori_size
    x_di   = transforms.Resize((rnd, rnd), interpolation=InterpolationMode.NEAREST)(x_adv)
    pad_max   = ori_size + di_pad_amount - rnd
    pad_left  = int(torch.rand(1) * pad_max)
    pad_right = pad_max - pad_left
    pad_top   = int(torch.rand(1) * pad_max)
    pad_bottom = pad_max - pad_top
    x_di = F.pad(x_di, (pad_left, pad_right, pad_top, pad_bottom), 'constant', 0)
    if img_width > 64:
        x_di = transforms.Resize((ori_size, ori_size), interpolation=InterpolationMode.NEAREST)(x_di)
    return x_di


# =============================================================================
# Gaussian kernel for TI
# =============================================================================
def gkern(kernlen=15, nsig=3):
    x          = np.linspace(-nsig, nsig, kernlen)
    kern1d     = st.norm.pdf(x)
    kernel_raw = np.outer(kern1d, kern1d)
    return kernel_raw / kernel_raw.sum()

In [6]:
filepath = '/content/FTM_Health/ftm/feature-tuning-mixup/main.py'

with open(filepath, 'r') as f:
    lines = f.readlines()

# Find the return parser line and insert new args just before it
new_args = '''\
    parser.add_argument("--epsilon",       type=int,   default=None,  help="Max L-inf perturbation 0-255")
    parser.add_argument("--alpha",         type=float, default=None,  help="Step size override")
    parser.add_argument("--mu",            type=float, default=1.0,   help="Momentum decay factor")
'''

new_lines = []
for line in lines:
    if 'return parser' in line:
        new_lines.append(new_args)   # insert before return
    new_lines.append(line)

with open(filepath, 'w') as f:
    f.writelines(new_lines)

# Now wire epsilon/alpha into exp_settings
with open(filepath, 'r') as f:
    code = f.read()

wire_code = """
    if args.epsilon is not None:
        exp_settings['epsilon'] = args.epsilon
    if args.alpha is not None:
        exp_settings['alpha'] = args.alpha
"""

# Insert after the beta line
code = code.replace(
    "exp_settings['ftm_beta']          = args.beta\n",
    "exp_settings['ftm_beta']          = args.beta\n" + wire_code
)

# Pass mu into ftm_attack
code = code.replace(
    "num_iter=exp_settings['max_iter']\n        )",
    "num_iter=exp_settings['max_iter'],\n            mu=args.mu\n        )"
)

with open(filepath, 'w') as f:
    f.write(code)

# Verify it worked
with open(filepath, 'r') as f:
    content = f.read()

checks = ['--epsilon', '--alpha', '--mu', "exp_settings['epsilon']"]
for c in checks:
    status = "✅" if c in content else "❌ MISSING"
    print(f"{status}  {c}")

✅  --epsilon
✅  --alpha
✅  --mu
✅  exp_settings['epsilon']


In [7]:
!find /content/FTM_Health -name "main.py" -type f -print

/content/FTM_Health/ftm/feature-tuning-mixup/main.py


In [8]:
%cd /content/FTM_Health/ftm/feature-tuning-mixup

/content/FTM_Health/ftm/feature-tuning-mixup


In [9]:
!python main.py \
  --attack_method RTMFNV \
  --num_images 250 \
  --epsilon 8 \
  --alpha 1.0 \
  --mu 1.0 \
  --ensemble_size 2 \
  --model_name xrv_chexnet \
  --target_models xrv_chexnet xrv_densenet_nih xrv_densenet_chex xrv_densenet_pc xrv_resnet50 \
  --eval

Execution Time: 2026-08-14|13:48:09
Arguments: Namespace(device='cuda:0', attack_method='RTMFNV', batch_size=2, model_name='xrv_chexnet', image_dir='./data/images/images-224', image_csv='./data/images/images-224.csv', save_dir='./exp/medical_results', beta=0.01, ensemble_size=2, eval=True, seed=42, img_size=224, config_idx=1, debug=False, num_images=250, max_iter=None, target_models=['xrv_chexnet', 'xrv_densenet_nih', 'xrv_densenet_chex', 'xrv_densenet_pc', 'xrv_resnet50'], epsilon=8, alpha=1.0, mu=1.0)
Config Settings: {'dataset': 'Medical-ChestXRay', 'targeted': True, 'epsilon': 8, 'alpha': 1.0, 'max_iter': 300, 'num_images': 250, 'p': 1.0, 'target_model_names': ['xrv_chexnet', 'xrv_densenet_nih', 'xrv_densenet_chex', 'xrv_densenet_pc', 'xrv_resnet50'], 'ftm_beta': 0.01, 'ftm_ensemble_size': 2, 'mix_prob': 0.1, 'mix_upper_bound_feature': 0.75, 'mixed_image_type_feature': 'C', 'shuffle_image_feature': 'SelfShuffle', 'blending_mode_feature': 'M', 'mix_lower_bound_feature': 0.0, 'diviso